# Bronze Ingestion

## Goal:
- create or use the landing volume
- copy the first raw order file into the landing area
- ingest raw data into a bronze Delta table

In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
config_path = "/Workspace/Users/adb-chonchanok@startsteps.org/de-mini-project-redcare-pharmacy/config/project_config.yml"

In [0]:
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
catalog_name = config["catalog"]
schema_name = config["schema"]

landing_volume = config["volumes"]["landing"]
checkpoint_volume = config["volumes"]["checkpoints"]
orders_subpath = config["paths"]["orders_subpath"]

landing_path = f"/Volumes/{catalog_name}/{schema_name}/{landing_volume}/{orders_subpath}"
checkpoint_path = f"/Volumes/{catalog_name}/{schema_name}/{checkpoint_volume}/orders_autoloader"

bronze_table = f"{catalog_name}.{schema_name}.{config['tables']['bronze_orders']}"

In [0]:
print(landing_path)
print(checkpoint_path)
print(bronze_table)

In [0]:
repo_sample_path = "/Workspace/Users/adb-chonchanok@startsteps.org/de-mini-project-redcare-pharmacy/sample_data"

In [0]:
dbutils.widgets.text("batch_file", "orders_batch1.csv")
batch_file = dbutils.widgets.get("batch_file")

print("Batch file: ", batch_file)

In [0]:
source_file = f"{repo_sample_path}/{batch_file}"
target_file = f"{landing_path}/{batch_file}"

dbutils.fs.cp(
    source_file,
    target_file
)

In [0]:
display(dbutils.fs.ls(landing_path))

In [0]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{target_file}")
)

display(raw_df)


In [0]:
raw_df.write.mode("overwrite").saveAsTable(bronze_table)